# Fusion Network Training Notebook

This notebook trains a fusion network using the M3DF dataset.

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm import tqdm
import argparse
from data import M3DF
from torch.utils.data import DataLoader

from data.M3DF import M3DF
from models.nets.FushionNet import FushionNet
from models.losses.FusionLoss import FusionLoss


data = M3DF(root="datasets/M3FD",train=True)

In [ ]:
loader = DataLoader(data, batch_size=32, shuffle=True, num_workers=2)
vis_list = []
for ir_Y, vis_Y,vis_Cb, vis_Cr,vis_list in loader:
    # ir_batch, vis_batch: (B, C, H, W); idx_batch: (B,) 用于对比学习.
    # vis_list: (B,) 用于对比学习.
    # ('03438.png',
    #  '00994.png',
    #  '00719.png',
    #  '00834.png',
    #   ...,
    # '00300.png')
    # 
    vis_list= vis_list
    print(ir_Y.shape)
    print(vis_Y.shape)
    print(vis_Cb.shape)
    print(vis_Cr.shape)
    break


## Define Training Parameters

In [ ]:
# Define default parameters
dataset_dir = 'datasets/M3FD'  # Path to dataset
batch_size = 16  # Batch size
epochs = 10  # Number of epochs
learning_rate = 1e-4  # Learning rate
num_workers = 4  # Number of workers for dataloader
save_dir = 'checkpoints'  # Directory to save checkpoints
save_interval = 5  # Save interval for checkpoints

## Training Function

In [ ]:
def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Dataset and DataLoader
    train_dataset = M3DF(root=dataset_dir, train=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    
    # Model Setup
    model = FushionNet(in_channels=1, out_channels=1, feat_channels=256).to(device)
    
    # Loss and Optimizer
    criterion = FusionLoss().to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    
    # Save directory
    os.makedirs(save_dir, exist_ok=True)
    
    # Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        
        # ir_Y, vis_Y, vis_Cb, vis_Cr, vis_list_index
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
        for ir, vis, cb, cr, _ in pbar:
            ir = ir.to(device)
            vis = vis.to(device)
            
            optimizer.zero_grad()
            fused = model(ir, vis)
            
            loss = criterion(fused, ir, vis)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.set_postf ix({'loss': f"{loss.item():.4f}"})
            
        print(f"Epoch [{epoch}/{epochs}], Average Loss: {epoch_loss/len(train_loader):.4f}")
        
        if epoch % save_interval == 0 or epoch == epochs:
            checkpoint_path = os.path.join(save_dir, f'fusion_model_epoch_{epoch}.pth')
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Saved checkpoint to {checkpoint_path}")
    
    return model

## Run Training

In [ ]:
# Execute the training
trained_model = train()
print("Training completed!")